In [1]:
"""
c5_orbit_structure.py
=====================
Explicit structure of the t-orbit in E.

We compute the monoid orbit generated by t under left multiplication
by all basis elements. List all 20 elements, group them by structure,
and look for a "2 × 10" decomposition.


By Néstor E. Ramos


"""

import numpy as np
from collections import defaultdict

MOD = 9
N = 9
FANO = [(0,1,2), (0,3,4), (0,5,6), (1,3,5), (1,4,6), (2,3,6), (2,4,5)]


def build(t_sq, t_left, t_right):
    M = np.zeros((N, N, N), dtype=int)
    for i in range(N):
        M[7, i, i] = 1
        M[i, 7, i] = 1
    for (a, b, c) in FANO:
        M[a, b, c] = 1
        M[b, c, a] = 1
        M[c, a, b] = 1
        M[b, a, c] = -1 % MOD
        M[c, b, a] = -1 % MOD
        M[a, c, b] = -1 % MOD
    for i in range(7):
        M[i, i, 7] = -1 % MOD
    M[8, 8, 7] = t_sq % MOD
    for i in range(7):
        M[8, i, i] = t_left % MOD
        M[i, 8, i] = t_right % MOD
    M[8, 7, 7] = t_left % MOD
    M[7, 8, 8] = 1
    return M


def left_mats(M):
    L = []
    for i in range(N):
        Lmat = np.zeros((N, N), dtype=int)
        for j in range(N):
            for k in range(N):
                Lmat[k, j] = M[i, j, k] % MOD
        L.append(Lmat)
    return L


def orbit(x, L):
    seen = set()
    stack = [tuple(int(v) % MOD for v in x)]
    while stack:
        v = stack.pop()
        if v in seen:
            continue
        seen.add(v)
        vv = np.array(v, dtype=int)
        for Lmat in L:
            new_v = tuple(int(w) % MOD for w in (Lmat @ vv))
            if new_v not in seen:
                stack.append(new_v)
    return seen


def label(v):
    parts = []
    for i, x in enumerate(v):
        if x == 0:
            continue
        if i < 7:
            parts.append(f"{x}·e_{i}")
        elif i == 7:
            parts.append(f"{x}·1")
        else:
            parts.append(f"{x}·t")
    return " + ".join(parts) if parts else "0"


def analyze(name, M):
    print("=" * 72)
    print(f"t-ORBIT: {name}")
    print("=" * 72)
    print()

    L = left_mats(M)
    t_vec = np.zeros(N, dtype=int); t_vec[8] = 1
    orb = orbit(t_vec, L)

    print(f"Total orbit size: {len(orb)}")
    print()

    # Group by support
    groups = defaultdict(list)
    for v in orb:
        nz_comps = tuple(i for i, x in enumerate(v) if x != 0)
        groups[nz_comps].append(v)

    print("Grouped by support (which components are non-zero):")
    for key in sorted(groups.keys(), key=lambda k: (len(k), k)):
        vecs = groups[key]
        comp_name = (
            "vacuum" if len(key) == 0 else
            ", ".join(f"e_{i}" if i < 7 else ("1" if i == 7 else "t") for i in key)
        )
        print(f"\n  support = {{{comp_name}}}  ({len(vecs)} element{'s' if len(vecs)>1 else ''})")
        for v in sorted(vecs)[:8]:
            print(f"    {label(v)}")
        if len(vecs) > 8:
            print(f"    ... and {len(vecs)-8} more")

    print()

    # Try "2 × 10" decompositions
    print("Searching for a 2 × 10 structure:")

    # Structure 1: split by t-component sign
    t_zero = [v for v in orb if v[8] == 0]
    t_nonzero = [v for v in orb if v[8] != 0]
    print(f"  t = 0:       {len(t_zero)} elements")
    print(f"  t ≠ 0:       {len(t_nonzero)} elements")
    print()

    # Structure 2: by negation pairing
    seen = set()
    neg_pairs = []
    self_neg = []
    for v in orb:
        if v in seen:
            continue
        neg = tuple((-x) % MOD for x in v)
        if neg == v:
            self_neg.append(v)
            seen.add(v)
        elif neg in orb:
            neg_pairs.append((v, neg))
            seen.add(v)
            seen.add(neg)
        else:
            self_neg.append(v)  # no pair, singleton
            seen.add(v)
    print(f"  Self-negative: {len(self_neg)}")
    print(f"  Negation pairs: {len(neg_pairs)}  (total {2*len(neg_pairs)})")
    print()

    # Structure 3: by component values
    print("Distribution of component values across all orbit elements:")
    for i in range(N):
        vals = [v[i] for v in orb]
        nonzero_vals = [x for x in vals if x != 0]
        if nonzero_vals:
            from collections import Counter
            c = Counter(nonzero_vals)
            comp_name = f"e_{i}" if i < 7 else ("1" if i == 7 else "t")
            print(f"  {comp_name:>3}: {dict(sorted(c.items()))}")
    print()

    return orb


M_E = build(t_sq=0, t_left=3, t_right=-3)
M_G = build(t_sq=-1, t_left=1, t_right=-1)

orb_E = analyze("E (drain)", M_E)
print()
orb_G = analyze("G (oscillator)", M_G)


# ============================================================
# SIDE-BY-SIDE COMPARISON
# ============================================================
print("=" * 72)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 72)
print()

print(f"{'Metric':>40} {'E':>10} {'G':>10}")
print("-" * 62)
print(f"{'Orbit size':>40} {len(orb_E):>10} {len(orb_G):>10}")

# Count by class
def count_multiple(v, k):
    return all(x % k == 0 for x in v)

E_mult3 = sum(1 for v in orb_E if count_multiple(v, 3))
G_mult3 = sum(1 for v in orb_G if count_multiple(v, 3))
print(f"{'Divisible by 3':>40} {E_mult3:>10} {G_mult3:>10}")

# Zero element
E_zero = 1 if any(all(x == 0 for x in v) for v in orb_E) else 0
G_zero = 1 if any(all(x == 0 for x in v) for v in orb_G) else 0
print(f"{'Contains zero':>40} {E_zero:>10} {G_zero:>10}")

# Purely t-element
E_t_only = sum(1 for v in orb_E if v[8] != 0 and all(v[i] == 0 for i in range(8)))
G_t_only = sum(1 for v in orb_G if v[8] != 0 and all(v[i] == 0 for i in range(8)))
print(f"{'Pure-t elements (only t non-zero)':>40} {E_t_only:>10} {G_t_only:>10}")

# Fano-only elements
E_fano = sum(1 for v in orb_E if all(v[i] == 0 for i in [7,8]) and any(v[i] != 0 for i in range(7)))
G_fano = sum(1 for v in orb_G if all(v[i] == 0 for i in [7,8]) and any(v[i] != 0 for i in range(7)))
print(f"{'Fano-only elements':>40} {E_fano:>10} {G_fano:>10}")

# Identity-only
E_1_only = sum(1 for v in orb_E if v[7] != 0 and all(v[i] == 0 for i in [0,1,2,3,4,5,6,8]))
G_1_only = sum(1 for v in orb_G if v[7] != 0 and all(v[i] == 0 for i in [0,1,2,3,4,5,6,8]))
print(f"{'Identity-only elements':>40} {E_1_only:>10} {G_1_only:>10}")

print()
print("=" * 72)
print("VERDICT")
print("=" * 72)
print()

# Analyze E specifically
E_t_nonzero = [v for v in orb_E if v[8] != 0]
E_t_zero = [v for v in orb_E if v[8] == 0]

print(f"E's orbit decomposes as:")
print(f"  t ≠ 0 subset: {len(E_t_nonzero)} elements")
print(f"  t = 0 subset: {len(E_t_zero)} elements")
print()

if len(E_t_nonzero) == 10 and len(E_t_zero) == 10:
    print("  ✓ Clean 10 + 10 split!")
    print("    → 2 × 10 structure confirmed.")
    print("    → Sub-channel = (10 + 10) / (10 × 10) = 20/100 = 0.20")
    print("    → C.5 is POSITIVE.")
elif len(E_t_zero) == 20:
    print("  All 20 elements have t = 0.")
    print("  → The orbit does not contain t itself in general.")
    print("  → Sub-channel interpretation needs revision.")
else:
    print(f"  Not a clean 10 + 10 split.")
    print(f"  E: {len(E_t_nonzero)} with t ≠ 0, {len(E_t_zero)} with t = 0")
    print(f"  Sub-channel = 20 / (10^2) = 0.20 if the alphabet is 10.")
    print(f"  → Interpretation depends on how the 20 elements partition.")

t-ORBIT: E (drain)

Total orbit size: 20

Grouped by support (which components are non-zero):

  support = {vacuum}  (1 element)
    0

  support = {e_0}  (2 elements)
    3·e_0
    6·e_0

  support = {e_1}  (2 elements)
    3·e_1
    6·e_1

  support = {e_2}  (2 elements)
    3·e_2
    6·e_2

  support = {e_3}  (2 elements)
    3·e_3
    6·e_3

  support = {e_4}  (2 elements)
    3·e_4
    6·e_4

  support = {e_5}  (2 elements)
    3·e_5
    6·e_5

  support = {e_6}  (2 elements)
    3·e_6
    6·e_6

  support = {1}  (2 elements)
    3·1
    6·1

  support = {t}  (3 elements)
    1·t
    3·t
    6·t

Searching for a 2 × 10 structure:
  t = 0:       17 elements
  t ≠ 0:       3 elements

  Self-negative: 2
  Negation pairs: 9  (total 18)

Distribution of component values across all orbit elements:
  e_0: {3: 1, 6: 1}
  e_1: {3: 1, 6: 1}
  e_2: {3: 1, 6: 1}
  e_3: {3: 1, 6: 1}
  e_4: {3: 1, 6: 1}
  e_5: {3: 1, 6: 1}
  e_6: {3: 1, 6: 1}
    1: {3: 1, 6: 1}
    t: {1: 1, 3: 1, 6: 1}


t-O